In [ ]:
# ============================================================
# Plot ESA WorldCover LULC (Bode, 30 m) + observation stations
#  - masks nodata background (no green outside basin)
#  - legend outside the map, no overlap
#  - station labels with white halo
# ============================================================

import numpy as np
import geopandas as gpd
import rioxarray as rxr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import Point
from matplotlib import patheffects as pe
from pathlib import Path

# ---------- PATHS ----------
LULC_FILE = Path(r"Y:\Home\goyal\post_doc_work\ecostress\data\lulc\WorldCover_Bode_2020_30m.tif")
BASIN_SH  = r"F:\goyal_shekhar\ecostress\data\shapefiles\bode_shp\bode.shp"

# ---------- STATIONS ----------
STATIONS = [
    ("Water_Steinerne_Renne", 51.81801424730135, 10.729251572843205),
    ("Water_Nienhagen",       51.94162487466206, 11.15862981180194),
    ("Water_Mahndorf",        51.88507046058312, 10.963237851032797),
    ("Water_Hausneindorf",    51.83886576185539, 11.269586522589968),
    ("Water_Meisdorf",        51.69143490670407, 11.283704018605757),
    ("Water_Silberhuette",    51.63249877292527, 11.097123452473651),
]

# ---------- LAND-COVER CLASSES & COLOURS (ESA WorldCover palette) ----------
LC_CLASSES = {
    10: ("Trees",        "#006400"),
    20: ("Shrubland",    "#ffbb22"),
    30: ("Grassland",    "#ffff4c"),
    40: ("Cropland",     "#f096ff"),
    50: ("Built-up",     "#fa0000"),
    60: ("Barren",       "#b4b4b4"),
    70: ("Snow/Ice",     "#f0f0f0"),
    80: ("Water",        "#0064c8"),
    90: ("Wetlands",     "#0096a0"),
    95: ("Mangroves",    "#00cf75"),
    100:("Moss/Lichen",  "#fae6a0"),
}
LC_CODES  = sorted(LC_CLASSES.keys())
LC_COLORS = [LC_CLASSES[c][1] for c in LC_CODES]

# ---------- PLOTTING STYLE ----------
plt.rcParams.update({
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
})

def map_only_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_visible(False)

def panel_letter(ax, letter):
    ax.text(
        0.01, 0.99, f"({letter})",
        transform=ax.transAxes,
        ha="left", va="top",
        fontweight="bold", fontsize=9,
        color="black",
    )

# ============================================================
# 1. LOAD DATA
# ============================================================
# LULC
lulc = rxr.open_rasterio(LULC_FILE).squeeze(drop=True)  # DataArray

if not lulc.rio.crs:
    lulc = lulc.rio.write_crs("EPSG:4326")

# Basin outline
basin = gpd.read_file(BASIN_SH).to_crs(lulc.rio.crs)

# Stations as GeoDataFrame
stations_gdf = gpd.GeoDataFrame(
    {
        "station": [s[0] for s in STATIONS],
        "lat": [s[1] for s in STATIONS],
        "lon": [s[2] for s in STATIONS],
    },
    geometry=[Point(s[2], s[1]) for s in STATIONS],  # (lon, lat)
    crs="EPSG:4326",
).to_crs(lulc.rio.crs)

# ============================================================
# 2. CLIP & MASK NODATA (background)
# ============================================================
lulc_clip = lulc.rio.clip(basin.geometry, basin.crs, drop=True)

# Mask nodata (WorldCover uses 0 as _FillValue)
nodata_val = 0
lulc_plot = lulc_clip.where(lulc_clip != nodata_val)

# Get x/y coords
if "x" in lulc_plot.coords and "y" in lulc_plot.coords:
    x = lulc_plot.x
    y = lulc_plot.y
elif "lon" in lulc_plot.coords and "lat" in lulc_plot.coords:
    x = lulc_plot["lon"]
    y = lulc_plot["lat"]
else:
    x_dim = list(lulc_plot.dims)[-1]
    y_dim = list(lulc_plot.dims)[-2]
    x = lulc_plot.coords[x_dim]
    y = lulc_plot.coords[y_dim]

extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]

# ============================================================
# 3. DISCRETE COLORMAP & NORM
# ============================================================
cmap = ListedColormap(LC_COLORS)
boundaries = list(np.array(LC_CODES) - 0.5) + [LC_CODES[-1] + 0.5]
norm = BoundaryNorm(boundaries=boundaries, ncolors=len(LC_CODES))

# ============================================================
# 4. PLOT MAP + STATIONS, LEGEND OUTSIDE
# ============================================================
fig = plt.figure(figsize=(4.5, 5.0))

# Axes slightly higher to leave space for legend
ax = fig.add_axes([0.05, 0.20, 0.90, 0.75])  # [left, bottom, width, height]

im = ax.imshow(
    lulc_plot.values,
    extent=extent,
    cmap=cmap,
    norm=norm,
    origin="upper",
)

# Basin outline
basin.boundary.plot(ax=ax, edgecolor="black", linewidth=0.7)

# Stations
stations_gdf.plot(
    ax=ax,
    markersize=20,
    color="white",
    edgecolor="black",
    linewidth=0.6,
    marker="o",
    zorder=5,
)

# Station labels with white halo
for _, row in stations_gdf.iterrows():
    txt = ax.text(
        row.geometry.x + (extent[1]-extent[0]) * 0.005,
        row.geometry.y + (extent[3]-extent[2]) * 0.005,
        row["station"].replace("Water_", ""),
        fontsize=6,
        color="black",
        ha="left",
        va="bottom",
        zorder=6,
    )
    txt.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

map_only_axes(ax)
panel_letter(ax, "a")
ax.set_title("ESA WorldCover (2020, 30 m)\nBode basin with observation stations", pad=4)

# ---------- Legend BELOW the map, outside axes ----------
lc_handles = [
    Patch(facecolor=LC_CLASSES[c][1], edgecolor="none", label=LC_CLASSES[c][0])
    for c in LC_CODES
]
station_handle = Line2D(
    [], [], marker="o", linestyle="None",
    markerfacecolor="white", markeredgecolor="black",
    label="Observation station", markersize=5,
)

legend = fig.legend(
    handles=lc_handles + [station_handle],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),  # outside axes, under plot
    ncol=4,
    frameon=False,
    fontsize=7,
)

fig.savefig(
    r"Y:\Home\goyal\post_doc_work\ecostress\figures\Bode_LULC_stations_30m_v2.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
